In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install spacy 

Note: you may need to restart the kernel to use updated packages.


In [3]:
!python -m spacy validate

✔ Loaded compatibility table

================ Installed pipeline packages (spaCy v3.8.11) ================
ℹ spaCy installation: /usr/local/lib/python3.12/dist-packages/spacy

NAME             SPACY            VERSION                            
en_core_web_sm   >=3.8.0,<3.9.0   3.8.0   ✔



In [4]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 39.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import spacy
nlp=spacy.load('en_core_web_md') #en_core_md is a small English model(13B)
print('spacy ready!')

spacy ready!


In [6]:
import re

def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',                      # MM/DD/YYYY or DD/MM/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',                      # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'                           # YYYY-MM-DD
    ]

    dates = []

    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)

    return dates

text = 'Invoice date: 03/15/2024. Due: March 30, 2024'
print(extract_dates(text))


['03/15/2024', 'March 30, 2024']


In [7]:
import re

def extract_amounts(text):
    pattern = r'\$?\d+(?:,\d{3})*(?:\.\d{2})?'
    
    amounts = re.findall(pattern, text)
    
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        cleaned.append(float(clean))
    
    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 112.45"
print(extract_amounts(text))

[1250.5, 125.05, 112.45]


In [8]:
import re

def extract_invoice(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if match.groups():
                return match.group(1)
            else:
                return match.group(0)

    return None

text = "Invoice Number: INV-2024-001"
print(extract_invoice(text))

INV-2024-001


In [9]:
import spacy
#load model
nlp=spacy.load('en_core_web_md')
#sample invoice 
text="""Invoice from Acme Corporation 123 Main 123 Main street,New York, NY 10001
contact: John Smith (john@acme.com) Date:March 15,2004, Amount Due: $1,250.50"""
doc=nlp(text)
print('Found Entities:')
for ent in doc.ents:
    print(f'{ent.text:20} {ent.label_:15} {spacy.explain(ent.label_)}')

Found Entities:
123                  CARDINAL        Numerals that do not fall under another type
New York             GPE             Countries, cities, states
NY 10001             ORG             Companies, agencies, institutions, etc.
John Smith           PERSON          People, including fictional
March 15,2004        DATE            Absolute or relative dates or periods
1,250.50             MONEY           Monetary values, including unit


In [10]:
def extract_entities(text):
    doc=nlp(text)
    entities={
        'persons':[],
        'organizations':[],
        'locations':[],
        'dates':[],
        'money':[]
    }
    for ent in doc.ents:
        if ent.label_ =='PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE','LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ =='DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
        
    return entities
    
results=extract_entities(text)
for entity_type, values in results.items():
    print(f'{entity_type}: {values}')

persons: ['John Smith']
organizations: ['NY 10001']
locations: ['New York']
dates: ['March 15,2004']
money: ['1,250.50']


In [11]:
from spacy import displacy

displacy.render(doc, style='ent', jupyter=True)

# Generate HTML for saving
html = displacy.render(doc, style='ent', page=True)

with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(str(html))

print("Visualization saved to entities.html")

Visualization saved to entities.html


In [14]:
import json
import pytesseract
from PIL import Image
import re
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# -------- REGEX FUNCTIONS --------
def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',
        r'\d{1,2}-\d{1,2}-\d{4}',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'
    ]
    dates = []
    for pattern in patterns:
        dates.extend(re.findall(pattern, text))
    return dates

def extract_amounts(text):
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        if clean:
            cleaned.append(float(clean))
    return cleaned

def extract_invoice_number(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# -------- NER FUNCTION --------
def extract_entities(text):
    doc = nlp(text)
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)

    return entities

# -------- MAIN FUNCTION --------
def process_invoice(image_path):
    """
    Complete pipeline:
    OCR → Extraction → JSON
    """
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)

    # Step 2: Regex extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }

    # Step 3: NER
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 4: Post-processing
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])

    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data

# -------- TEST --------
result = process_invoice('/kaggle/input/datasets/musfiranazahat/ye-looo/Gemini_Generated_Image_huotvvhuotvvhuot.png')
print(json.dumps(result, indent=2))

{
  "invoice_number": null,
  "dates": [
    "80",
    "33H",
    "33u4 3909"
  ],
  "amounts": [
    6.0,
    8.0,
    66.0,
    18.0,
    41.0,
    91.0,
    391.0,
    1.0,
    3.0,
    0.0,
    391.0,
    1.0,
    1.0,
    41.0,
    80.0,
    11.0,
    34.0,
    2.0,
    114.0,
    33.0,
    39.0,
    110.0,
    9.0,
    47.0,
    90.0,
    91.0,
    44.0,
    1.0,
    1.0,
    33.0,
    4.0,
    390.0,
    9.0
  ],
  "persons": [
    "Vd",
    "S.YTILLYOL"
  ],
  "organizations": [
    "Sb",
    "PTUIOJST TED"
  ],
  "locations": [],
  "money": [
    "6\u00b0 XUL",
    "#8#",
    "41"
  ],
  "total_amount": 391.0,
  "invoice_date": "80"
}


In [ ]:
import json

output_file = 'extracted_data.json'

with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f"Results saved to {output_file}")



### **Lab 7 Summary: Information Extraction & Named Entity Recognition**

This lab focuses on converting unstructured text into structured data using **Regular Expressions (Regex)** and **Named Entity Recognition (NER)**. The main objective is to extract useful information such as dates, amounts, invoice numbers, names, organizations, and locations from documents.

In the first part, regex techniques are used to identify patterns in text. Functions are created to extract:

* Dates in multiple formats
* Currency amounts and convert them into numeric values
* Invoice or order numbers using predefined patterns

In the second part, the **spaCy library** is used for Named Entity Recognition. It helps identify entities like:

* Persons
* Organizations
* Locations
* Dates
* Monetary values

The extracted entities are also visualized using **displaCy**, which provides a color-coded representation of text data.

In the final part, a complete pipeline is built by combining:

1. **OCR (Optical Character Recognition)** to extract text from images
2. **Regex** for pattern-based extraction
3. **NER** for intelligent entity detection

The processed data is then structured into **JSON format**, making it easy to store and analyze.

Overall, this lab demonstrates how to build a simple document intelligence system that converts raw text or images into meaningful structured information.

